# Principal Components Analysis (PCA)

## Learning Objectives
- Preprocess data: **mean-centre** and **variance-normalise** each feature
- Derive that the direction maximising projected variance is the **principal eigenvector** of the sample covariance $\Sigma$
- Generalise to $k$ principal components as the top-$k$ eigenvectors of $\Sigma$
- Implement PCA from scratch and verify against `sklearn`
- Understand **variance explained** as a guide for choosing $k$

## Problem Statement

Given data $\{x^{(1)},\ldots,x^{(m)}\}$ with $x^{(i)} \in \mathbb{R}^n$, find a $k$-dimensional subspace ($k \ll n$) that **maximises the variance** of the projected data.

**Preprocessing (4 steps):**
1. Mean-centre: $x^{(i)} \leftarrow x^{(i)} - \mu$, where $\mu = \frac{1}{m}\sum_i x^{(i)}$
2. Variance-normalise: $x^{(i)}_j \leftarrow x^{(i)}_j / \sigma_j$, where $\sigma_j^2 = \frac{1}{m}\sum_i(x^{(i)}_j)^2$ (omit if features already on same scale)

**Variance maximisation.** For unit vector $u$, projected variance is:
$$\frac{1}{m}\sum_i \bigl((x^{(i)})^T u\bigr)^2 = u^T \Sigma u, \quad \Sigma = \frac{1}{m}\sum_i x^{(i)}{x^{(i)}}^T$$

**Objective:** $\max_{\|u\|=1}\; u^T \Sigma u$

By Lagrange multipliers, $\Sigma u = \lambda u$ — so $u$ must be an **eigenvector** of $\Sigma$, and the maximum is achieved by the **eigenvector with the largest eigenvalue**.

**For $k$ components:** choose the top-$k$ eigenvectors $u_1,\ldots,u_k$. The reduced representation is:
$$y^{(i)} = \begin{bmatrix} u_1^T x^{(i)} \\ \vdots \\ u_k^T x^{(i)} \end{bmatrix} \in \mathbb{R}^k$$

## 1. Variance Maximisation: Geometric Intuition

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# 2D dataset with clear principal direction
cov_true = np.array([[3.0, 2.0], [2.0, 1.5]])
X = np.random.multivariate_normal([0, 0], cov_true, 200)

# Compute PCA by hand
Sigma = X.T @ X / len(X)
eigvals, eigvecs = np.linalg.eigh(Sigma)
order = eigvals.argsort()[::-1]
eigvals, eigvecs = eigvals[order], eigvecs[:, order]
u1, u2 = eigvecs[:, 0], eigvecs[:, 1]

# Project onto several directions and compare variance
angles = np.linspace(0, np.pi, 180)
variances = [(np.array([np.cos(a), np.sin(a)]) @ Sigma @ np.array([np.cos(a), np.sin(a)]))
             for a in angles]

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# Left: data with principal components
ax = axes[0]
ax.scatter(X[:, 0], X[:, 1], s=15, alpha=0.4, c='steelblue')
scale = 2.5
for ev, u, c, lbl in [(eigvals[0], u1, '#d6604d', f'PC1 (λ={eigvals[0]:.2f})'),
                       (eigvals[1], u2, '#4dac26', f'PC2 (λ={eigvals[1]:.2f})')
                      ]:
    ax.annotate('', xy=scale*np.sqrt(ev)*u, xytext=-scale*np.sqrt(ev)*u,
                arrowprops=dict(arrowstyle='->', color=c, lw=2.5, mutation_scale=18))
    ax.text(*(scale*np.sqrt(ev)*u + 0.15), lbl, color=c, fontsize=9, fontweight='bold')
ax.set_aspect('equal')
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.set_title('Principal components = eigenvectors of $\\Sigma$\n(length ∝ √eigenvalue)')

# Right: variance vs projection direction
ax = axes[1]
ax.plot(np.degrees(angles), variances, 'b-', lw=2.5)
pc1_angle = np.degrees(np.arctan2(u1[1], u1[0])) % 180
pc2_angle = np.degrees(np.arctan2(u2[1], u2[0])) % 180
ax.axvline(pc1_angle, color='#d6604d', ls='--', lw=2,
           label=f'PC1 direction: max var={eigvals[0]:.2f}')
ax.axvline(pc2_angle, color='#4dac26', ls='--', lw=2,
           label=f'PC2 direction: min var={eigvals[1]:.2f}')
ax.set_xlabel('Projection direction (degrees)')
ax.set_ylabel('Projected variance $u^T \\Sigma u$')
ax.set_title('Variance as a function of projection direction\nMaximum at PC1')
ax.legend(fontsize=9)

fig.suptitle('PCA: Principal Direction = Maximum-Variance Eigenvector',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. PCA from Scratch and Preprocessing

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA as skPCA

def pca(X, k):
    """PCA: returns top-k principal components and projected coordinates."""
    mu    = X.mean(axis=0)
    Xc    = X - mu
    Sigma = Xc.T @ Xc / len(Xc)
    eigvals, eigvecs = np.linalg.eigh(Sigma)
    order   = eigvals.argsort()[::-1]
    eigvals = eigvals[order]
    U       = eigvecs[:, order]         # columns = eigenvectors
    U_k     = U[:, :k]                  # top-k
    Y       = Xc @ U_k                  # projected coords (m, k)
    var_exp = eigvals / eigvals.sum()
    return U_k, Y, eigvals, var_exp, mu

np.random.seed(7)

# 4D dataset with mixed scales (preprocessing matters)
n_feat = 4
scales = np.array([100.0, 1.0, 50.0, 0.5])   # features on very different scales
cov4   = np.array([[1.0, 0.8, 0.3, 0.1],
                   [0.8, 1.0, 0.2, 0.4],
                   [0.3, 0.2, 1.0, 0.6],
                   [0.1, 0.4, 0.6, 1.0]])
X4_raw = np.random.multivariate_normal(np.zeros(n_feat), cov4, 300) * scales

# Without normalisation
U_no, Y_no, ev_no, vexp_no, _ = pca(X4_raw, 2)

# With normalisation
sigma_j = X4_raw.std(axis=0)
X4_norm = X4_raw / sigma_j
U_nrm, Y_nrm, ev_nrm, vexp_nrm, _ = pca(X4_norm, 2)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Left: eigenvalues without normalisation
ax = axes[0]
ax.bar(range(n_feat), ev_no, color='#d6604d', edgecolor='k', alpha=0.8,
       label='Without normalisation')
ax.bar(range(n_feat), ev_nrm, color='#2166ac', edgecolor='k', alpha=0.5,
       label='With normalisation')
ax.set_xlabel('Principal component'); ax.set_ylabel('Eigenvalue')
ax.set_title('Eigenvalue spectrum\nNormalisation prevents scale domination')
ax.legend(fontsize=8.5)

# Middle: cumulative variance explained
ax = axes[1]
ax.plot(range(1, n_feat+1), np.cumsum(vexp_no),  'r-o', lw=2, ms=7, label='Without normalisation')
ax.plot(range(1, n_feat+1), np.cumsum(vexp_nrm), 'b-o', lw=2, ms=7, label='With normalisation')
ax.axhline(0.95, color='k', ls='--', lw=1.5, label='95% threshold')
ax.set_xlabel('Number of components $k$')
ax.set_ylabel('Cumulative variance explained')
ax.set_title('Variance explained vs $k$\n(elbow = good choice of $k$)')
ax.legend(fontsize=8.5)

# Right: 2D projection
ax = axes[2]
ax.scatter(Y_nrm[:, 0], Y_nrm[:, 1], s=20, alpha=0.5, c='#2166ac', label='Normalised')
ax.scatter(Y_no[:, 0] / Y_no[:, 0].std(), Y_no[:, 1] / Y_no[:, 1].std(),
           s=20, alpha=0.5, c='#d6604d', label='Unnormalised (scaled)')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title('2D PCA projection')
ax.legend(fontsize=8.5)

# Compare with sklearn
pca_sk = skPCA(n_components=2)
Y_sk   = pca_sk.fit_transform(X4_norm)
corr1  = abs(np.corrcoef(Y_nrm[:, 0], Y_sk[:, 0])[0, 1])
corr2  = abs(np.corrcoef(Y_nrm[:, 1], Y_sk[:, 1])[0, 1])
print(f'Agreement with sklearn — PC1: {corr1:.6f}, PC2: {corr2:.6f}')

fig.suptitle('PCA Implementation — Preprocessing Effects', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Reconstruction and Approximation Error

PCA projects onto a $k$-dimensional subspace. The **reconstruction** of $x^{(i)}$ from its projections:
$$\hat{x}^{(i)} = \mu + \sum_{j=1}^k y^{(i)}_j \, u_j$$

The mean squared reconstruction error equals the sum of the **discarded eigenvalues**:
$$\frac{1}{m}\sum_i \|x^{(i)} - \hat{x}^{(i)}\|^2 = \sum_{j=k+1}^n \lambda_j$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(3)

# 10D dataset, project to k=2 and reconstruct
n_d, m_d = 10, 500
cov_d = np.diag([5, 3, 2, 1.5, 1, 0.8, 0.5, 0.3, 0.2, 0.1])
X_d = np.random.multivariate_normal(np.zeros(n_d), cov_d, m_d)

U_all, _, ev_all, vexp_all, mu_d = pca(X_d, n_d)

# Reconstruction error vs k
k_vals = range(1, n_d + 1)
recon_errors, theory_errors = [], []
for k in k_vals:
    U_k   = U_all[:, :k]
    Xc    = X_d - mu_d
    Y_k   = Xc @ U_k
    X_hat = mu_d + Y_k @ U_k.T
    recon_errors.append(np.mean(np.sum((X_d - X_hat)**2, axis=1)))
    theory_errors.append(ev_all[k:].sum())   # sum of discarded eigenvalues

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(k_vals, recon_errors, 'b-o', lw=2.5, ms=6, label='Empirical MSE')
ax.plot(k_vals, theory_errors, 'r--s', lw=2, ms=6, label='$\\sum_{j>k}\\lambda_j$ (theory)')
ax.set_xlabel('Number of components $k$')
ax.set_ylabel('Reconstruction MSE')
ax.set_title('Reconstruction error = sum of discarded eigenvalues')
ax.legend(fontsize=9)

ax = axes[1]
ax.plot(k_vals, np.cumsum(vexp_all) * 100, 'g-o', lw=2.5, ms=6)
for thresh, ls in [(90, '--'), (95, '-.'), (99, ':')]:
    ax.axhline(thresh, color='k', ls=ls, lw=1.5, label=f'{thresh}%')
    k_needed = next(k for k, v in enumerate(np.cumsum(vexp_all)*100, 1) if v >= thresh)
    ax.axvline(k_needed, color='gray', ls=ls, lw=1, alpha=0.6)
ax.set_xlabel('Number of components $k$')
ax.set_ylabel('Cumulative variance explained (%)')
ax.set_title('Choose $k$ by variance threshold')
ax.legend(fontsize=9)

fig.suptitle('PCA Reconstruction Error and Variance Explained', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Eigenfaces: PCA for Face Images

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_olivetti_faces
from sklearn.decomposition import PCA as skPCA

faces = fetch_olivetti_faces(shuffle=True, random_state=42)
X_faces = faces.data   # (400, 4096) — 400 images, 64x64 pixels
n_imgs, n_pixels = X_faces.shape
img_shape = (64, 64)

k_faces = 40
pca_f = skPCA(n_components=k_faces, random_state=0)
Y_faces  = pca_f.fit_transform(X_faces)
var_face = pca_f.explained_variance_ratio_

fig, axes = plt.subplots(3, 6, figsize=(15, 8))

# Row 1: original faces
for j in range(6):
    axes[0, j].imshow(X_faces[j].reshape(img_shape), cmap='gray')
    axes[0, j].axis('off')
axes[0, 0].set_title('Original', fontsize=9)

# Row 2: first 6 eigenfaces
for j in range(6):
    ef = pca_f.components_[j].reshape(img_shape)
    axes[1, j].imshow(ef, cmap='gray')
    axes[1, j].axis('off')
    axes[1, j].set_title(f'EF {j+1} ({var_face[j]*100:.1f}%)', fontsize=8)

# Row 3: reconstructed at k=5, 10, 20, 40 and 2 originals for reference
idx = 0
for j, k in enumerate([2, 5, 10, 20, 40, n_pixels]):
    if k < n_pixels:
        pca_tmp = skPCA(n_components=k, random_state=0)
        Y_tmp   = pca_tmp.fit_transform(X_faces)
        X_rec   = pca_tmp.inverse_transform(Y_tmp)
        img = X_rec[idx].reshape(img_shape)
        lbl = f'k={k}'
    else:
        img = X_faces[idx].reshape(img_shape)
        lbl = 'Original'
    axes[2, j].imshow(img, cmap='gray')
    axes[2, j].axis('off')
    axes[2, j].set_title(lbl, fontsize=8)

axes[0, 0].set_ylabel('Originals', fontsize=9)
axes[1, 0].set_ylabel('Eigenfaces', fontsize=9)
axes[2, 0].set_ylabel('Reconstructed', fontsize=9)

fig.suptitle(f'Eigenfaces: PCA on {n_imgs} Face Images ({n_pixels}-dim → {k_faces}-dim)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Variance explained by top {k_faces} components: {var_face.sum()*100:.1f}%')
print(f'Each face = weighted sum of {k_faces} eigenfaces (instead of {n_pixels} raw pixels)')

## 5. Derivation Pathway

### Derivation pathway

In [ ]:
from IPython.display import HTML
HTML("""
<svg xmlns="http://www.w3.org/2000/svg" width="780" height="468"
     viewBox="0 0 780 468" font-family="Segoe UI, Arial, sans-serif">
  <defs>
    <marker id="ah" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#444"/>
    </marker>
    <marker id="ahd" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#999"/>
    </marker>
  </defs>
  <rect x="10" y="12" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="35" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">High-dim data</text>
  <text x="102" y="52" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">x&#x2071; &#x2208; &#x211d;&#x207f;</text>
  <line x1="197" y1="35" x2="216" y2="35"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="12" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="40" font-size="13" text-anchor="middle" fill="#333"
        >want to find k &#x226a; n directions that retain most variance</text>
  <line x1="102" y1="58" x2="102" y2="120"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="80" font-size="11.5" font-style="italic" fill="#555"
        >centre: x&#x2071; := x&#x2071; &#x2212; &#x3bc;</text>
  <rect x="10" y="112" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="140" font-size="13.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Centred data</text>
  <line x1="197" y1="135" x2="216" y2="135"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="112" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="140" font-size="13" text-anchor="middle" fill="#333"
        >empirical covariance &#x3a3; = (1/m) &#x2211;&#x2071; x&#x2071;x&#x2071;&#x1d40;</text>
  <line x1="102" y1="158" x2="102" y2="220"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="180" font-size="11.5" font-style="italic" fill="#555"
        >eigendecompose &#x3a3; = U&#x39b;U&#x1d40;</text>
  <rect x="10" y="212" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="235" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Eigenvectors</text>
  <text x="102" y="252" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">u&#x2081;,...,u&#x2099;</text>
  <line x1="197" y1="235" x2="216" y2="235"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="212" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="240" font-size="13" text-anchor="middle" fill="#333"
        >sort by eigenvalue &#x3bb;&#x2081; &#x2265; &#x3bb;&#x2082; &#x2265; ... &#x2014; principal components</text>
  <line x1="102" y1="258" x2="102" y2="320"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="280" font-size="11.5" font-style="italic" fill="#555"
        >project: z&#x2071; = U&#x1d40;&#x2096;x&#x2071; (top k cols of U)</text>
  <rect x="10" y="312" width="185" height="46" rx="7"
        fill="#1a5fa8" stroke="#1a5fa8" stroke-width="2"/>
  <text x="102" y="335" font-size="12.5" font-weight="700"
        text-anchor="middle" fill="#ffffff">PCA embedding</text>
  <text x="102" y="352" font-size="12.5" font-weight="700"
        text-anchor="middle" fill="#ffffff">z&#x2071; &#x2208; &#x211d;&#x1d55c;</text>
  <line x1="197" y1="335" x2="216" y2="335"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="312" width="548" height="46" rx="7"
        fill="#dce8f8" stroke="#7aadd4" stroke-width="1.5"/>
  <text x="495" y="340" font-size="13" text-anchor="middle" fill="#333"
        >k-dim representation; reconstruct: x&#x303;&#x2071; = U&#x2096;z&#x2071;; error = &#x2211;&#x2c7;&#x227e;&#x2096; &#x3bb;&#x2c7;</text>
</svg>
""")

## Summary

| Concept | Formula | Key Insight |
|---|---|---|
| Preprocessing | $x^{(i)} := x^{(i)} - \mu$ (and optionally scale) | Essential before PCA |
| Empirical covariance | $\Sigma = \frac{1}{m}\sum_i x^{(i)}{x^{(i)}}^T$ | PCA finds eigenvectors of $\Sigma$ |
| Principal components | Top $k$ eigenvectors $u_1,\ldots,u_k$ | Directions of maximum variance |
| Projection | $z^{(i)} = U_k^T x^{(i)} \in \mathbb{R}^k$ | Low-dimensional embedding |
| Reconstruction | $\tilde{x}^{(i)} = U_k z^{(i)}$ | Best rank-$k$ approximation (Eckart-Young) |
| Retained variance | $\sum_{j=1}^k \lambda_j / \sum_{j=1}^n \lambda_j$ | Fraction of total variance explained |

**Key insight:** PCA finds the $k$-dimensional subspace that minimises reconstruction error, which is equivalent to maximising retained variance; the solution is given by the top $k$ eigenvectors of the sample covariance matrix.